## Setup

In [ ]:
import sys
import os
import importlib

from tabulate import tabulate
import matplotlib.pyplot as plt
import pandas as pd

sys.path.append("../src")

import utils
import plot
import rstats
import mappings

# Reload modules to apply any changes
importlib.reload(utils)
importlib.reload(plot)
importlib.reload(rstats)
importlib.reload(mappings)

# FIXME: must have latex installed in this path ("/Library/TeX/texbin")
os.environ["PATH"] += os.pathsep + "/Library/TeX/texbin"
plt.rcParams["text.usetex"] = True
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Computer Modern Roman"]
plt.rcParams["figure.dpi"] = 300

base = 12
plt.rcParams.update(
    {
        "font.size": base,
        "axes.titlesize": base + 2,
        "axes.labelsize": base + 1,
        "xtick.labelsize": base - 1,
        "ytick.labelsize": base - 1,
        "legend.fontsize": base - 1,
        "figure.titlesize": base + 4,
    }
)

In [ ]:
FILENAME = os.getenv("FILENAME", "UFABC_PLT_combined")
BACKEND = os.getenv("BACKEND", "gemini")
DST = f"../results/{BACKEND}/{FILENAME}/"

print(f"{FILENAME=}")
print(f"{BACKEND=}")

df = utils.load(f"../results/{BACKEND}/{FILENAME}/metrics.csv")
print(f"{df.shape=}")

# Avoid log(0) issues
eps = 1e-10
df.loc[df.entropy <= 0, "entropy"] = eps
df.loc[df.distance_next <= 0, "distance_next"] = eps
df.loc[df.vel_magnitude <= 0, "vel_magnitude"] = eps
df.loc[df.acc_magnitude <= 0, "acc_magnitude"] = eps

In [ ]:
tmp = ["id", "category", "concept"] if "category" in df.columns else ["id", "concept"]
grouped = df.groupby(tmp, as_index=False)
dfx = grouped.mean(numeric_only=True)
print(tabulate(dfx.head(3), headers="keys", showindex=False))

## Analysis

In [ ]:
import io
import contextlib

metrics = [
    "distance_next",
    "vel_magnitude",
    "acc_magnitude",
    "entropy",
    "distance_centroid_static",
]

if "category" not in dfx.columns:
    dfx = dfx.rename(columns={"concept": "category"})

dfx["category"] = dfx["category"].map(mappings.categories.get(FILENAME))
cats = sorted(pd.Series(dfx["category"].unique()).tolist())

gridspec_kw = {
    "height_ratios": [1, 1],
    "hspace": 0.35,
    # "wspace": 0.1
}
figsize = mappings.figsize.get(FILENAME)
fig, axes = plt.subplots(2, len(metrics), figsize=figsize, gridspec_kw=gridspec_kw)

for i, metric in enumerate(metrics):
    print(f"[{i + 1}/{len(metrics)}] Analyzing '{metric}'")

    # Use lognormal for all metrics, except for distance_centroid_static
    family = "lognormal" if metric != "distance_centroid_static" else "gaussian"

    stdout = io.StringIO()
    with contextlib.redirect_stdout(stdout):
        # Fit GLMM and get emmeans + Tukey pairs
        glmm = rstats.glmm(dfx, formula=f"{metric} ~ category + (1|id)", family=family)
        pred = rstats.emmeans(effect="category")
        pairs = rstats.pairs()

    # Standardize column names for EMMs and plit "A - B" contrasts into separate columns
    if "emmean" not in pred and "response" in pred:
        pred = pred.rename(columns={"response": "emmean"})
    if "SE" not in pred and "SE.df" in pred:
        pred = pred.rename(columns={"SE.df": "SE"})
    if "contrast" in pairs.columns and not {"group1", "group2"}.issubset(pairs.columns):
        pairs[["group1", "group2"]] = pairs["contrast"].str.split(" - ", expand=True)

    # Save R output
    with open(f"{DST}/r-output-{metric}.txt", "w") as fp:
        fp.write(stdout.getvalue())

    # Individual figure: boxplot
    # ax1 = plot.boxplot(dfx, metric, pred, pairs, cats=cats, ax=None, figsize=mappings.figsize.get(FILENAME))
    # ax1.set_title(mappings.ylabels.get(metric, metric))
    # plt.tight_layout()
    # plt.savefig(f"{DST}/boxplot-{metric}.png", bbox_inches="tight")
    # plt.close()

    # Combined figure: boxplot
    ax2 = plot.boxplot(dfx, metric, pred, pairs, cats=cats, ax=axes[0, i])
    ax2.set_title(mappings.ylabels.get(metric, metric))

    # Combined figure: heatmap
    hm, pmat = plot.heatmap(cats=cats, pairs=pairs, ax=axes[1, i])
    pmat.to_csv(f"{DST}/pvalues-{metric}.csv")

cax = fig.add_axes([0.25, -0.05, 0.5, 0.03])
cbar = fig.colorbar(
    hm.collections[0],
    cax=cax,
    orientation="horizontal",
    ticks=[5e-5, 5.5e-4, 5.5e-3, 3e-2, 0.55],
)
cbar.ax.set_xticklabels(
    # ["p < 1e4 (****)", "p < 1e3 (***)", "p < 1e2 (**)", "p < 5e1 (*)", "ns"]
    ["$p < 1e-4$", "$p < 1e-3$", "$p < 1e-2$", "$p < 5e-2$", "ns"]
)

tmp = {
    "UFABC_PLT_combined": "ufabc",
    "Swear_fluency": "swearwords",
    "CPN120": "cpn120",
    "Dados_Italian_2": "italian",
    "German_data": "german",
    "Parkinson_paper": "parkinson",
}
tmp = tmp.get(FILENAME)

fig.savefig(f"{DST}/{BACKEND.lower()}-{tmp}.pdf", dpi=300, bbox_inches="tight")
plt.close()